In [1]:
from pyspark.sql.functions import col
from datetime import datetime

print("Incremental Load Notebook Started")

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 3, Finished, Available, Finished, False)

Incremental Load Notebook Started


In [2]:
lakehouse_name = "lh_dev_ecommerce"
env_name = "dev"

print(lakehouse_name)
print(env_name)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 4, Finished, Available, Finished, False)

lh_dev_ecommerce
dev


In [3]:
control_df = spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.control_table
WHERE IsActive = 'Yes'
""")

control_df.show(truncate=False)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 5, Finished, Available, Finished, False)

+-----------+-----------+-----------+---------------+--------+
|TableName  |PrimaryKey |LoadType   |WatermarkColumn|IsActive|
+-----------+-----------+-----------+---------------+--------+
|order_items|OrderItemID|Incremental|InsertTimestamp|Yes     |
|orders     |OrderID    |Incremental|InsertTimestamp|Yes     |
|customers  |CustomerID |Full       |InsertTimestamp|Yes     |
|products   |ProductID  |Full       |InsertTimestamp|Yes     |
+-----------+-----------+-----------+---------------+--------+



In [4]:
watermark_df = spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.watermark_table
""")

watermark_df.show(truncate=False)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 6, Finished, Available, Finished, False)

+-----------+-------------------+
|TableName  |LastLoadTimestamp  |
+-----------+-------------------+
|products   |1900-01-01 00:00:00|
|order_items|1900-01-01 00:00:00|
|customers  |1900-01-01 00:00:00|
|orders     |1900-01-01 00:00:00|
+-----------+-------------------+



In [5]:
control_dict = {}

for row in control_df.collect():

    control_dict[row["TableName"]] = {
        "PrimaryKey": row["PrimaryKey"],
        "LoadType": row["LoadType"],
        "WatermarkColumn": row["WatermarkColumn"]
    }

print(control_dict)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 7, Finished, Available, Finished, False)

{'order_items': {'PrimaryKey': 'OrderItemID', 'LoadType': 'Incremental', 'WatermarkColumn': 'InsertTimestamp'}, 'orders': {'PrimaryKey': 'OrderID', 'LoadType': 'Incremental', 'WatermarkColumn': 'InsertTimestamp'}, 'customers': {'PrimaryKey': 'CustomerID', 'LoadType': 'Full', 'WatermarkColumn': 'InsertTimestamp'}, 'products': {'PrimaryKey': 'ProductID', 'LoadType': 'Full', 'WatermarkColumn': 'InsertTimestamp'}}


In [6]:
watermark_dict = {}

for row in watermark_df.collect():

    watermark_dict[row["TableName"]] = row["LastLoadTimestamp"]

print(watermark_dict)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 8, Finished, Available, Finished, False)

{'products': datetime.datetime(1900, 1, 1, 0, 0), 'order_items': datetime.datetime(1900, 1, 1, 0, 0), 'customers': datetime.datetime(1900, 1, 1, 0, 0), 'orders': datetime.datetime(1900, 1, 1, 0, 0)}


In [7]:
table_name = "orders"

last_watermark = watermark_dict[table_name]

print(f"Table          : {table_name}")
print(f"Last Watermark : {last_watermark}")

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 9, Finished, Available, Finished, False)

Table          : orders
Last Watermark : 1900-01-01 00:00:00


In [8]:
incremental_df = spark.sql(f"""
SELECT *
FROM {lakehouse_name}.{env_name}_bronze_db.orders
WHERE InsertTimestamp > TIMESTAMP('{last_watermark}')
""")

incremental_df.show(truncate=False)

print("Rows Loaded :", incremental_df.count())

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 10, Finished, Available, Finished, False)

+-------+----------+----------+-----------+-------------+------------------+------+------------+----------+-------------------------+
|OrderID|CustomerID|OrderDate |ShippedDate|DeliveredDate|Price             |Region|OrderChannel|ReferredBy|InsertTimestamp          |
+-------+----------+----------+-----------+-------------+------------------+------+------------+----------+-------------------------+
|ORD001 |CUST008   |2023/02/04|2023/02/05 |2023/02/08   |163.77000000000001|South |Offline     |None      |2026-07-10 11:59:51.62932|
|ORD002 |CUST024   |2023/10/20|2023/10/23 |2023/10/27   |318.69999999999999|West  |Offline     |Facebook  |2026-07-10 11:59:51.62932|
|ORD003 |CUST012   |2023/08/05|2023/08/07 |2023/08/11   |877.45000000000005|North |Mobile App  |Google    |2026-07-10 11:59:51.62932|
|ORD004 |CUST004   |2023/05/08|2023/05/09 |2023/05/11   |582.65999999999997|South |Offline     |Facebook  |2026-07-10 11:59:51.62932|
|ORD005 |CUST018   |2023/02/20|2023/02/21 |2023/02/26   |852.4

In [9]:
from pyspark.sql.functions import max

new_watermark = incremental_df.select(
    max("InsertTimestamp")
).collect()[0][0]

print("New Watermark :", new_watermark)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 11, Finished, Available, Finished, False)

New Watermark : 2026-07-10 11:59:51.629320


In [10]:
spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.watermark_table
WHERE TableName = 'orders'
""").show(truncate=False)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 12, Finished, Available, Finished, False)

+---------+-------------------+
|TableName|LastLoadTimestamp  |
+---------+-------------------+
|orders   |1900-01-01 00:00:00|
+---------+-------------------+



In [11]:
from pyspark.sql.functions import max

for table_name, config in control_dict.items():

    print("=" * 60)
    print(f"Processing : {table_name}")

    load_type = config["LoadType"]
    watermark_column = config["WatermarkColumn"]

    if load_type == "Incremental":

        last_watermark = watermark_dict[table_name]

        print(f"Last Watermark : {last_watermark}")

        df = spark.sql(f"""
        SELECT *
        FROM {lakehouse_name}.{env_name}_bronze_db.{table_name}
        WHERE {watermark_column} > TIMESTAMP('{last_watermark}')
        """)

    else:

        print("Full Load")

        df = spark.sql(f"""
        SELECT *
        FROM {lakehouse_name}.{env_name}_bronze_db.{table_name}
        """)

    print("Rows Loaded :", df.count())

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 13, Finished, Available, Finished, False)

Processing : order_items
Last Watermark : 1900-01-01 00:00:00
Rows Loaded : 37
Processing : orders
Last Watermark : 1900-01-01 00:00:00
Rows Loaded : 25
Processing : customers
Full Load
Rows Loaded : 26
Processing : products
Full Load
Rows Loaded : 25


In [12]:
from pyspark.sql.functions import max

for table_name, config in control_dict.items():

    if config["LoadType"] == "Incremental":

        df = spark.sql(f"""
        SELECT *
        FROM {lakehouse_name}.{env_name}_bronze_db.{table_name}
        WHERE {config["WatermarkColumn"]} >
              TIMESTAMP('{watermark_dict[table_name]}')
        """)

        if df.count() > 0:

            new_watermark = df.select(
                max(config["WatermarkColumn"])
            ).collect()[0][0]

            spark.sql(f"""
            UPDATE {env_name}_metadata_db.watermark_table
            SET LastLoadTimestamp =
                TIMESTAMP('{new_watermark}')
            WHERE TableName = '{table_name}'
            """)

            print(f"{table_name} Watermark Updated")

        else:

            print(f"{table_name} : No New Records")

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 14, Finished, Available, Finished, False)

order_items Watermark Updated
orders Watermark Updated


In [13]:
spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.watermark_table
""").show(truncate=False)

StatementMeta(, 63d34139-717e-4bc8-972e-b27e2e1a8a8a, 15, Finished, Available, Finished, False)

+-----------+--------------------------+
|TableName  |LastLoadTimestamp         |
+-----------+--------------------------+
|products   |1900-01-01 00:00:00       |
|order_items|2026-07-10 11:59:34.799694|
|customers  |1900-01-01 00:00:00       |
|orders     |2026-07-10 11:59:51.62932 |
+-----------+--------------------------+

